# 136 — Arquitectura percepción-planificación-acción

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("robotics", seed=136)
assert result["kind"] == "robotics"
assert result["evidence"]
show(result)


## Solución 1 — Traza del aspirador

Dirección inicial derecha: paso 1 `C→D`... pero en el paso 3 el obstáculo aún
no existe, así que conviene trazar con cuidado:

1. `C→D` (avanza), 2. `D→E` (avanza), 3. limpia `E` (obstáculo aparece en `D`),
4. pared a la derecha → invierte; obstáculo en `D` delante → sigue bloqueado y
espera/invierte según la regla 1 (queda oscilando en `E`),
6. el obstáculo desaparece → `E→D`, 7. `D→C`, 8. `C→B`, 9. `B→A`, 10. limpia `A`.

Movimientos de traslación: 7 (C→D, D→E, E→D, D→C, C→B, B→A y la espera no
cuenta). El óptimo deliberativo sin obstáculo era 6. La lección: el reactivo
paga pocos movimientos extra pero nunca requiere replanificación global; el
número exacto puede variar ±1 según cómo definas la regla de espera — lo
importante es justificar tu convención.


In [ ]:
secuencia = ["C", "D", "E", "E*", "D", "C", "B", "A"]  # E* = espera por obstáculo
movimientos = 7
print(secuencia, movimientos)


## Solución 2 — Clasificación de fallas

- (a) **Percepción**: la estimación del estado (pared fantasma) no corresponde
  al mundo real.
- (b) **Planificación**: la información estaba disponible en el mapa; el
  planificador generó un plan inválido con un modelo correcto.
- (c) **Acción**: error de ejecución/actuación (holgura, calibración del
  motor); el comando era correcto.
- (d) **Feedback**: el resultado de la acción nunca vuelve al ciclo, así que el
  error se repite — es la definición de operar en lazo abierto.


## Solución 3 — Semillas y contrato

Al cambiar la semilla cambian los valores internos del episodio simulado
(decisiones muestreadas, métricas), pero **no** cambian las claves del
contrato: `kind` sigue siendo `"robotics"` y `evidence`/`limitations` siguen
presentes. El contrato estable es lo que permite validar automáticamente 180
laboratorios distintos: los tests dependen de la estructura, no de valores que
varían con la semilla.


In [ ]:
from ai_evolution.labs import run_lab

r1 = run_lab("robotics", seed=136)
r2 = run_lab("robotics", seed=1234)
assert r1["kind"] == r2["kind"] == "robotics"
assert r1["evidence"] and r2["evidence"]
print("claves:", sorted(r1.keys()))


## Solución 4 — Capas de la arquitectura híbrida

- (a) Bumper de emergencia → **reactiva**, cientos de Hz: la seguridad nunca
  puede esperar al planificador.
- (b) Replanificar por calle cerrada → **ejecutiva/deliberativa**, del orden de
  1 Hz o bajo demanda: requiere el mapa global.
- (c) Seguimiento de carril a 200 Hz → **reactiva**: lazo de control continuo
  con latencia mínima.
- (d) Orden de pedidos del día → **deliberativa**, minutos: optimización global
  sin restricción de tiempo real.
